## Project Description

This is an Abstract Syntax Tree (AST) analysis project created by Khalid Mihlar.

### Project Goal

I am analyzing student submissions from Assignment 3 of the CS 2420 class. All student submissions are de-identified and anonymous. The primary goal is to examine how the `size` variable, located within the `ArrayCollection` function (which extends `Collection` in Java), is interacted with, updated, and mutated across different functions. I will be creating an AST and analyzing each node to identify these interactions.


## Imports and Such
The main goal fo this block is to load in the proper location for the student submissions and then simply confirm what files were found and the name of the files

In [11]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple

import javalang
import pandas as pd

# Set this to the folder that contains your .java files.
# "." means the same folder as the notebook's current working directory.
ROOT_DIR = Path("./inputs/submissions")
JAVA_GLOB = "*.java"

if not ROOT_DIR.exists():
    raise FileNotFoundError(f"Folder does not exist: {ROOT_DIR.resolve()}")

java_files = sorted(ROOT_DIR.glob(JAVA_GLOB))

print(f"Looking for Java files in: {ROOT_DIR.resolve()}")
print("Found files:")
for p in java_files:
    print(" -", p.name)

if len(java_files) == 0:
    print("\nNo .java files found.")
    print("Put your five selected Java files in the folder above,")
    print("or change ROOT_DIR to the correct folder path.")
else:
    print(f"\nConfirmed {len(java_files)} Java file(s) found.")

Looking for Java files in: /Users/khalidmihlar/Code/ast-project/inputs/submissions
Found files:
 - AfricanWildDog.java
 - Anthozoa.java
 - BlackRhinoceros.java
 - Newt.java
 - Yak.java

Confirmed 5 Java file(s) found.


## This section is focused on taking the files found in a folder previously and then attempt to see if all the files can be parsed into a tree and saved into raw_trees


In [12]:
def load_source(path: Path) -> str:
    return path.read_text(encoding="utf-8")

def parse_java_file(path: Path):
    src = load_source(path)
    return javalang.parse.parse(src)

raw_trees = {}
parse_errors = {}

for path in java_files:
    try:
        raw_trees[path.name] = parse_java_file(path)
        print(f"Parsed successfully: {path.name}")
    except Exception as e:
        parse_errors[path.name] = str(e)
        print(f"Failed to parse: {path.name}")
        print(f"Error: {e}")
        print("-" * 50)



Parsed successfully: AfricanWildDog.java
Parsed successfully: Anthozoa.java
Parsed successfully: BlackRhinoceros.java
Parsed successfully: Newt.java
Parsed successfully: Yak.java


# Leftover Code
This was here for some understanding, don't want to delete it just in case I need it, but it had functionality for me to understand what was going on with some of the nodes

In [13]:
import javalang
import json

def dump_nodes(tree):
    for path, node in tree:
        print("=" * 60)
        print("NODE TYPE:", type(node).__name__)
        print("ATTRS:", getattr(node, "attrs", []))

        for attr in getattr(node, "attrs", []):
            print(f"  {attr}: {getattr(node, attr)}")

# Simply a way for me to see all the nodes and it's relevant information to get a better understanding 
def visualize_ast(node, indent=0):
    prefix = "  " * indent

    if isinstance(node, javalang.ast.Node):
        print(f"{prefix}{type(node).__name__}")

        for attr in node.attrs:
            value = getattr(node, attr)
            if value is None or value == []:
                continue

            print(f"{prefix}  .{attr}:")
            visualize_ast(value, indent + 2)

    elif isinstance(node, list):
        print(f"{prefix}list[{len(node)}]")
        for i, item in enumerate(node):
            print(f"{prefix}  [{i}]")
            visualize_ast(item, indent + 2)

    else:
        print(f"{prefix}{repr(node)}")

java_code = """
public class Example {
    public int test() {
        int size = 0;
        size++;
        return size;
    }
}
"""

print(raw_trees.keys())


dict_keys(['AfricanWildDog.java', 'Anthozoa.java', 'BlackRhinoceros.java', 'Newt.java', 'Yak.java'])


## Dataclass
This is where the AnalyzeRow data structure is defined for .csv population at the end


In [32]:
from dataclasses import dataclass, field

@dataclass
class AnalysisRow:
    file_name: str
    class_name: str
    method_name: str
    kind: str
    line_numbers: list[int] = field(default_factory=list)
    tags: list[str] = field(default_factory=list)
    metadata: str = ""

## Size and Constructor Helper Functions
Simple Ones

analyze_size_function_node

analyze_constructor_function_node

In [ ]:
# public int size()
def analyze_size_function_node(node, file_name):
    tags = []
    line_numbers = []

    if node.position is not None:
        line_numbers.append(node.position.line)

    if len(node.body) == 1:
        stmt = node.body[0]

        if isinstance(stmt, javalang.tree.ReturnStatement):
            if stmt.position is not None:
                line_numbers.append(stmt.position.line)

            expr = stmt.expression

            if isinstance(expr, javalang.tree.MemberReference):
                if expr.member == "size":
                    tags.append("returns_size")

            elif isinstance(expr, javalang.tree.This):
                tags.append("returns_this_size")

    if len(tags) == 0:
        tags.append("NULL")

    return AnalysisRow(
        file_name=file_name,
        class_name="ArrayCollection",
        method_name="size",
        kind="method",
        line_numbers=line_numbers,
        tags=tags,
        metadata=str(node)
    )

def analyze_constructor_function_node(node, file_name):
    tags = []
    line_numbers = []

    if node.position is not None:
        line_numbers.append(node.position.line)

    size_set_to_zero = False

    for path, inner_node in node:
        if isinstance(inner_node, javalang.tree.Assignment):
            left_side = inner_node.expressionl
            right_side = inner_node.value

            if (
                isinstance(left_side, javalang.tree.MemberReference)
                and left_side.member == "size"
                and isinstance(right_side, javalang.tree.Literal)
                and right_side.value == "0"
            ):
                size_set_to_zero = True

                if inner_node.position is not None:
                    line_numbers.append(inner_node.position.line)

    if size_set_to_zero:
        tags.append("sets_size_to_0")

    if len(tags) == 0:
        tags.append("NULL")

    return AnalysisRow(
        file_name=file_name,
        class_name="ArrayCollection",
        method_name="ArrayCollection",
        kind="constructor",
        line_numbers=sorted(set(line_numbers)),
        tags=tags,
        metadata=str(node)
    )

[AnalysisRow(file_name='AfricanWildDog.java', class_name='ArrayCollection', method_name='ArrayCollection', kind='constructor', line_numbers=[24], tags=['sets_size_to_0'], metadata='ConstructorDeclaration(annotations=[Annotation(element=Literal(postfix_operators=[], prefix_operators=[], qualifier=None, selectors=[], value="unchecked"), name=SuppressWarnings)], body=[StatementExpression(expression=Assignment(expressionl=MemberReference(member=size, postfix_operators=[], prefix_operators=[], qualifier=, selectors=[]), type==, value=Literal(postfix_operators=[], prefix_operators=[], qualifier=None, selectors=[], value=0)), label=None), StatementExpression(expression=Assignment(expressionl=MemberReference(member=index, postfix_operators=[], prefix_operators=[], qualifier=, selectors=[]), type==, value=Literal(postfix_operators=[], prefix_operators=[], qualifier=None, selectors=[], value=0)), label=None), StatementExpression(expression=Assignment(expressionl=MemberReference(member=data, post

## Add function helper and stuff


In [50]:
def contains_size(expr):
    if isinstance(expr, javalang.tree.MemberReference):
        return expr.member == "size"
    if isinstance(expr, javalang.tree.BinaryOperation):
        return contains_size(expr.operandl) or contains_size(expr.operandr)
    return False


def contains_length(expr):
    if isinstance(expr, javalang.tree.MemberReference):
        return expr.member == "length"
    if isinstance(expr, javalang.tree.BinaryOperation):
        return contains_length(expr.operandl) or contains_length(expr.operandr)
    return False


def is_size_plus_one(expr):
    return (
        isinstance(expr, javalang.tree.BinaryOperation)
        and expr.operator == "+"
        and (
            (
                isinstance(expr.operandl, javalang.tree.MemberReference)
                and expr.operandl.member == "size"
                and isinstance(expr.operandr, javalang.tree.Literal)
                and expr.operandr.value == "1"
            )
            or
            (
                isinstance(expr.operandr, javalang.tree.MemberReference)
                and expr.operandr.member == "size"
                and isinstance(expr.operandl, javalang.tree.Literal)
                and expr.operandl.value == "1"
            )
        )
    )

def contains_size_function_call(expr):
    if isinstance(expr, javalang.tree.MethodInvocation):
        return expr.member == "size" and len(expr.arguments) == 0
    if isinstance(expr, javalang.tree.BinaryOperation):
        return contains_size_function_call(expr.operandl) or contains_size_function_call(expr.operandr)
    return False


def analyze_add_function_node(node, file_name):
    tags = []
    line_numbers = []
    metadata_parts = []

    if node.position is not None:
        line_numbers.append(node.position.line)

    grow_called = False
    grow_called_with_capacity_check = False

    size_updated = False
    standard_increment = False
    nonstandard_size_change = False

    array_insert_uses_size = False

    for path, inner_node in node:
        if getattr(inner_node, "position", None) is not None:
            line_numbers.append(inner_node.position.line)

        # detect grow anywhere in method
        if isinstance(inner_node, javalang.tree.MethodInvocation) and inner_node.member == "grow":
            grow_called = True

        # ---------- grow checks ----------
        if isinstance(inner_node, javalang.tree.IfStatement):
            condition = inner_node.condition

            if isinstance(condition, javalang.tree.BinaryOperation):
                left = condition.operandl
                right = condition.operandr
                operator = condition.operator

                then_stmt = inner_node.then_statement
                found_grow_in_if = False

                if then_stmt is not None:
                    for _, then_node in then_stmt:
                        if (
                            isinstance(then_node, javalang.tree.MethodInvocation)
                            and then_node.member == "grow"
                        ):
                            found_grow_in_if = True
                            break

                if found_grow_in_if:
                    grow_called = True
                    metadata_parts.append(f"grow condition: {condition}")

                    # size == length  OR  length == size
                    if (
                        operator == "=="
                        and contains_size(left)
                        and contains_length(right)
                    ) or (
                        operator == "=="
                        and contains_length(left)
                        and contains_size(right)
                    ):
                        grow_called_with_capacity_check = True
                        tags.append("grow_check_equal")

                    # size >= length  OR  length <= size
                    elif (
                        operator == ">="
                        and contains_size(left)
                        and contains_length(right)
                    ) or (
                        operator == "<="
                        and contains_length(left)
                        and contains_size(right)
                    ):
                        grow_called_with_capacity_check = True
                        tags.append("grow_check_goe")

                    # length < size + 1  OR  size + 1 > length
                    elif (
                        operator == "<"
                        and contains_length(left)
                        and is_size_plus_one(right)
                    ) or (
                        operator == ">"
                        and is_size_plus_one(left)
                        and contains_length(right)
                    ):
                        grow_called_with_capacity_check = True
                        tags.append("grow_check_plus_one")

                    # some other size/length-based grow condition
                    elif contains_size(condition) and contains_length(condition):
                        grow_called_with_capacity_check = True
                        tags.append("grow_check_other")

                    # length == size()  OR  size() == length
                    elif (
                        operator == "=="
                        and contains_length(left)
                        and contains_size_function_call(right)
                    ) or (
                        operator == "=="
                        and contains_size_function_call(left)
                        and contains_length(right)
                    ):
                        grow_called_with_capacity_check = True
                        tags.append("growth_check_with_size_function")

        # ---------- array insert using size / assignment-based size updates ----------
        if isinstance(inner_node, javalang.tree.Assignment):
            left_side = inner_node.expressionl

            if isinstance(left_side, javalang.tree.MemberReference):
                # size = ...
                if left_side.member == "size":
                    size_updated = True

                    if inner_node.type == "=":
                        value = inner_node.value

                        if isinstance(value, javalang.tree.BinaryOperation):
                            if (
                                isinstance(value.operandl, javalang.tree.MemberReference)
                                and value.operandl.member == "size"
                                and isinstance(value.operandr, javalang.tree.Literal)
                                and value.operandr.value == "1"
                                and value.operator == "+"
                            ):
                                standard_increment = True
                                metadata_parts.append(f"size update: {inner_node}")
                            else:
                                nonstandard_size_change = True
                                metadata_parts.append(f"nonstandard size update: {inner_node}")
                        else:
                            nonstandard_size_change = True
                            metadata_parts.append(f"nonstandard size update: {inner_node}")

                    elif inner_node.type == "+=":
                        value = inner_node.value
                        if isinstance(value, javalang.tree.Literal) and value.value == "1":
                            standard_increment = True
                            metadata_parts.append(f"size update: {inner_node}")
                        else:
                            nonstandard_size_change = True
                            metadata_parts.append(f"nonstandard size update: {inner_node}")

                    else:
                        nonstandard_size_change = True
                        metadata_parts.append(f"nonstandard size update: {inner_node}")

                # check data[...] = ... where index uses size
                if left_side.member == "data" and left_side.selectors:
                    for selector in left_side.selectors:
                        if isinstance(selector, javalang.tree.ArraySelector):
                            index = selector.index

                            if contains_size(index):
                                array_insert_uses_size = True
                                metadata_parts.append(f"array insert: {left_side}")

        # ---------- size++ / ++size / size-- / --size ----------
        if isinstance(inner_node, javalang.tree.MemberReference) and inner_node.member == "size":
            if inner_node.postfix_operators or inner_node.prefix_operators:
                size_updated = True

                if "++" in (inner_node.postfix_operators or []) or "++" in (inner_node.prefix_operators or []):
                    standard_increment = True
                    metadata_parts.append(f"size update: {inner_node}")
                elif "--" in (inner_node.postfix_operators or []) or "--" in (inner_node.prefix_operators or []):
                    nonstandard_size_change = True
                    metadata_parts.append(f"nonstandard size update: {inner_node}")

    # ---------- finalize tags ----------
    if not grow_called:
        tags.append("grow_missing")
    elif grow_called and not grow_called_with_capacity_check:
        tags.append("grow_called_without_capacity_check")

    if array_insert_uses_size:
        tags.append("array_insert_uses_size")

    if standard_increment:
        tags.append("increment_size")
    elif nonstandard_size_change:
        tags.append("size_change_nonstandard")
    elif not size_updated:
        tags.append("size_no_update")
        metadata_parts.append("no size update detected in add method")

    if len(tags) == 0:
        tags.append("NULL")

    return AnalysisRow(
        file_name=file_name,
        class_name="ArrayCollection",
        method_name="add",
        kind="method",
        line_numbers=sorted(set(line_numbers)),
        tags=sorted(set(tags)),
        metadata=" | ".join(metadata_parts) if metadata_parts else str(node)
    )

## Main Function to run AST Node Analysis to produce List of AnalysisRows

In [51]:
def process_node(node, file_name):
    if (
        isinstance(node, javalang.tree.MethodDeclaration)
        and node.name == "size"
        and len(node.parameters) == 0
    ):
        return analyze_size_function_node(node, file_name)
    elif (
        isinstance(node, javalang.tree.ConstructorDeclaration)
        and node.name == "ArrayCollection"
        and len(node.parameters) == 0
    ):
        return analyze_constructor_function_node(node, file_name)
    elif (
        isinstance(node, javalang.tree.MethodDeclaration)
        and node.name == "add"
        and len(node.parameters) == 1
    ):
        return analyze_add_function_node(node, file_name)

    return None

def analyze_all_submissions(raw_trees):
    all_rows = []
    for file_name, tree in raw_trees.items():
        for path, node in tree:
            result = process_node(node, file_name)
            if result is not None:
                all_rows.append(result)
    return all_rows

info = analyze_all_submissions(raw_trees)
print(info)

[AnalysisRow(file_name='AfricanWildDog.java', class_name='ArrayCollection', method_name='ArrayCollection', kind='constructor', line_numbers=[24], tags=['sets_size_to_0'], metadata='ConstructorDeclaration(annotations=[Annotation(element=Literal(postfix_operators=[], prefix_operators=[], qualifier=None, selectors=[], value="unchecked"), name=SuppressWarnings)], body=[StatementExpression(expression=Assignment(expressionl=MemberReference(member=size, postfix_operators=[], prefix_operators=[], qualifier=, selectors=[]), type==, value=Literal(postfix_operators=[], prefix_operators=[], qualifier=None, selectors=[], value=0)), label=None), StatementExpression(expression=Assignment(expressionl=MemberReference(member=index, postfix_operators=[], prefix_operators=[], qualifier=, selectors=[]), type==, value=Literal(postfix_operators=[], prefix_operators=[], qualifier=None, selectors=[], value=0)), label=None), StatementExpression(expression=Assignment(expressionl=MemberReference(member=data, post

## Converting to a .csv
This will take the finalized listed data structure of AnalysisRow and process the information into a .csv file that is easily readable


In [52]:
import csv

def write_analysis_rows_to_csv(info, output_file="analysis_output.csv"):
    with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)

        writer.writerow([
            "File Name",
            "Class Name",
            "Method Name",
            "Kind",
            "Line Numbers",
            "Tags",
            "Metadata"
        ])

        for row in info:
            writer.writerow([
                row.file_name,
                row.class_name,
                row.method_name,
                row.kind,
                ", ".join(str(num) for num in row.line_numbers),
                ", ".join(row.tags),
                row.metadata
            ])

    print(f"CSV written to {output_file}")

write_analysis_rows_to_csv(info)

CSV written to analysis_output.csv
